# Notebook 03: Preamble Extraction Proposal

## Background and Objectives
This notebook implements the Task 3 preamble extraction proposal across all 193 unique national constitutions.

Following the diagnostic findings of Notebook 02—which revealed that 36 constitutions in the dataset contain full-body legislative articles rather than genuine preambles, and identified HTML entity corruption across multiple records—this pipeline establishes a transparent, rule-based extraction methodology:
1. **Entity and Whitespace Normalization**: Decodes HTML entities (`html.unescape`), resolves unescaped/broken entity remnants (`x27;`, `quot;`, `nbsp;`, `amp;`, stray `gt;`/`lt;`), and collapses multiple spaces and newlines into a single space.
2. **Rule-Based Extraction & Status Assignment**:
   - `no_preamble_in_source`: Constitutions lacking a preamble in the source text (starting immediately with structural headings like Chapter/Article/Part and exceeding 1,000 words). The proposal sets `extracted_text = ""` (0 words) to avoid feeding body text into preamble models.
   - `long_cut`: Constitutions where a genuine preamble is followed immediately by the constitution body (exceeding 1,000 words without a structural start). We locate the first structural heading after word 30 and isolate the preamble.
   - `short_keep`: Standard preambles (<= 400 words, non-structural start). Kept intact.
   - `medium_review`: Preambles between 401 and 1,000 words without structural start. Kept intact for review.
   - `manual_review`: Borderline cases requiring individual inspection (e.g., structural starts under 1,000 words, or long texts without a heading after word 30).
3. **Audit Reports**: Detailed breakdowns per status, crosstabulations against original cluster assignments, cut boundaries, and extracted texts under 30 words.
4. **Data Deliverable**: Generating `data/processed/preamble_extraction_proposal.csv` with columns: `Country, orig_cluster, orig_words, status, rule, extracted_words, extracted_text`.

In [1]:
import os
import re
import html
import json
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

data_dir = '../data/processed'
clustered_csv_path = os.path.join(data_dir, 'constitution_preamble_clustered.csv')
cleaned_csv_path = os.path.join(data_dir, 'constitution_dataset_cleaned.csv')
proposal_csv_path = os.path.join(data_dir, 'preamble_extraction_proposal.csv')

print(f"Data directory: {os.path.abspath(data_dir)}")


Data directory: /home/devpanchal/Projects/PreambleMap/data/processed


## 1. Data Ingestion and Dataset Alignment

We load `data/processed/constitution_preamble_clustered.csv` (which contains original cluster assignments) and `data/processed/constitution_dataset_cleaned.csv` (which contains raw Preamble text).
- **Cyprus**: Duplicate row 192 (an identical duplicate of row 44) is dropped, yielding exactly 193 unique countries.
- **Slovakia**: The missing raw text (`NaN`) is populated in-memory from Notebook 01 Cell 31 (`slovak_republic` string).

In [2]:
df_clust = pd.read_csv(clustered_csv_path)
df_clean = pd.read_csv(cleaned_csv_path)

# Drop Cyprus duplicate row 192
df = df_clust.drop(index=192).reset_index(drop=True)
df['Preamble'] = df_clean.drop(index=192).reset_index(drop=True)['Preamble']

# Load Slovakia text from Notebook 01 Cell 31
with open('01_data_preprocessing.ipynb', 'r', encoding='utf-8') as f:
    nb01 = json.load(f)
loc = {}
exec(''.join(nb01['cells'][31]['source']), {'df': pd.DataFrame([{'Country': 'Slovakia', 'Preamble': None}])}, loc)
df.loc[df['Country'] == 'Slovakia', 'Preamble'] = loc['slovak_republic']

print(f"Total rows ingested: {len(df)} countries")
print(f"Cyprus occurrences: {len(df[df['Country'] == 'Cyprus'])}")
print(f"Slovakia Preamble length (chars): {len(df.loc[df['Country'] == 'Slovakia', 'Preamble'].values[0])}")


Total rows ingested: 193 countries
Cyprus occurrences: 1
Slovakia Preamble length (chars): 120730


## 2. Step 1: Entity Cleanup and Normalization

Before assigning status or performing cuts, we apply entity and whitespace normalization across all 193 rows:
1. `html.unescape` decodes standard HTML entities.
2. Unescaped/broken entity remnants are repaired:
   - `x27;` -> `'` (apostrophe)
   - `quot;` -> `"` (quotation mark)
   - `nbsp;` -> ` ` (space)
   - `amp;` -> `&` (ampersand)
   - Stray `gt;` and `lt;` are removed using word boundaries (`\bgt;`, `\blt;`) so words like *guilt;*, *silt;*, and *salt;* are preserved.
3. Whitespace characters and newlines are collapsed to single spaces (`\s+` -> `' '`) and stripped.

In [3]:
def clean_entities(text):
    if pd.isna(text):
        return ''
    t = html.unescape(str(text))
    t = t.replace('x27;', "'").replace('quot;', '"').replace('nbsp;', ' ').replace('amp;', '&')
    t = re.sub(r'\bgt;', ' ', t)
    t = re.sub(r'\blt;', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

raw_texts = df['Preamble'].tolist()
cleaned_texts = [clean_entities(t) for t in raw_texts]
df['cleaned_raw'] = cleaned_texts
df['orig_words'] = df['cleaned_raw'].apply(lambda x: len(x.split()))
df['orig_cluster'] = df['cluster']

changed_rows = sum(1 for r, c in zip(raw_texts, cleaned_texts) if str(r) != c)
print(f"Total rows modified by Step 1 entity/whitespace cleanup: {changed_rows}")

print("\n=== 5 BEFORE / AFTER ENTITY CLEANUP EXAMPLES (WITH CONTEXT) ===")
sample_countries = ['Afghanistan', 'Andorra', 'Bangladesh', 'Central African Rep.', 'New Zealand']
for sc in sample_countries:
    raw_val = df.loc[df['Country'] == sc, 'Preamble'].values[0]
    cl_val = df.loc[df['Country'] == sc, 'cleaned_raw'].values[0]
    
    if sc == 'Afghanistan':
        pat = 'x27;'
    elif sc == 'Andorra':
        pat = 'quot;'
    elif sc == 'Bangladesh':
        pat = 'x27;'
    elif sc == 'Central African Rep.':
        pat = 'quot;'
    elif sc == 'New Zealand':
        pat = 'nbsp;'
    
    idx = raw_val.find(pat)
    start = max(0, idx - 30)
    end_raw = min(len(raw_val), idx + len(pat) + 30)
    snippet_before = raw_val[start:end_raw]
    
    prefix = raw_val[max(0, idx - 20):idx]
    cl_idx = cl_val.find(prefix)
    snippet_after = cl_val[max(0, cl_idx - 10):cl_idx + len(prefix) + 35]
    
    print(f"\nCountry: {sc} (Target Entity: {pat})")
    print(f"  BEFORE: {repr(snippet_before)}")
    print(f"  AFTER:  {repr(snippet_after)}")


Total rows modified by Step 1 entity/whitespace cleanup: 67

=== 5 BEFORE / AFTER ENTITY CLEANUP EXAMPLES (WITH CONTEXT) ===

Country: Afghanistan (Target Entity: x27;)
  BEFORE: ' of the martyrs of the countryx27;s freedom; Comprehending that '
  AFTER:  " of the martyrs of the country's freedom; Comprehending that a un"

Country: Andorra (Target Entity: quot;)
  BEFORE: 'ions, Desiring that the motto quot;virtus, unita, fortiorquot;, w'
  AFTER:  'ions, Desiring that the motto "virtus, unita, fortior", which has'

Country: Bangladesh (Target Entity: x27;)
  BEFORE: ' independent, sovereign Peoplex27;s Republic of Bangladesh; Pled'
  AFTER:  " independent, sovereign People's Republic of Bangladesh; Pledging"

Country: Central African Rep. (Target Entity: quot;)
  BEFORE: ' respect for the principle of quot;Zo Kwe Zoquot; declared by the'
  AFTER:  ' respect for the principle of "Zo Kwe Zo" declared by the Foundin'

Country: New Zealand (Target Entity: nbsp;)
  BEFORE: 'Legislature Act

## 3. Step 2: Status Assignment Heuristic

We evaluate each country using the extended `starts_with_structure` heuristic and word count bands:

1. `starts_with_structure`: Checks the first 12 words of the text for:
   - Structural keywords: `Part`, `Chapter`, `Title`, `Article`, `Section`, `Book`, `Schedule`, `Preliminary`.
   - Roman numerals: `I`, `II`, `III`, `IV`, etc.
   - Extended multi-word structural headings: `Fundamental Principles`, `Basic Law`, `General Provisions`, `Instrument of Government`, `Constitution Act`.

2. **Precedence Hierarchy**:
   1. `no_preamble_in_source`: `starts_with_structure is True` and `words > 1000`. Source text contains only constitution body articles. `extracted_text = ""` and `extracted_words = 0`.
   2. `long_cut`: `not starts_with_structure` and `words > 1000`. Locates the first structural heading (`Chapter`, `Title`, `Part`, `Article`, `Art`, `Section`, with optional numeral/Roman numeral) occurring AFTER word 30. The text is cut immediately before this heading. If none is found or the extract would be under 30 words, status is set to `manual_review`.
   3. `short_keep`: `not starts_with_structure` and `words <= 400`. Kept intact.
   4. `medium_review`: `not starts_with_structure` and `401 <= words <= 1000`. Kept intact for review.
   5. `manual_review`: Everything else (e.g. structural start with words <= 1000, or long texts without cut heading).

The resulting extraction proposal is saved to `data/processed/preamble_extraction_proposal.csv`.

In [4]:
extended_phrases = [
    'fundamental principles', 'basic law', 'general provisions',
    'instrument of government', 'constitution act'
]
single_keywords = {
    'part', 'chapter', 'title', 'article', 'section', 'book', 'schedule', 'preliminary'
}
roman_numerals = {'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii'}

def check_structure_start(text):
    words = text.split()
    first_12 = words[:12]
    first_12_str = ' '.join(first_12).lower()
    has_phrase = any(phrase in first_12_str for phrase in extended_phrases)
    has_kw = any(re.sub(r'^[^\w]+|[^\w]+$', '', w).lower() in single_keywords for w in first_12)
    has_rm = any(re.sub(r'^[^\w]+|[^\w]+$', '', w).lower() in roman_numerals for w in first_12)
    return has_phrase or has_kw or has_rm

numeral_or_roman = r'(?:[0-9]+|[IVXLCDMivxlcdm]+|One|Two|Three|Four|Five|Six|Seven|Eight|Nine|Ten|First|Second|Third|Fourth|Fifth)'
heading_re = re.compile(
    r'\b(Chapter|Title|Part|Article|Art|Section)(?:\s+' + numeral_or_roman + r'\b|\b(?!\s+(?:of|in|and|the|a|an|to|for|with|by|from|or|meant)\b))',
    re.IGNORECASE
)

status_list = []
rule_list = []
extracted_text_list = []
extracted_words_list = []

for i, r in df.iterrows():
    txt = r['cleaned_raw']
    words = txt.split()
    n_words = len(words)
    is_struct = check_structure_start(txt)
    
    if is_struct and n_words > 1000:
        st = 'no_preamble_in_source'
        rl = 'structural_start_and_words_gt_1000'
        ext_txt = ''
        ext_w = 0
    elif (not is_struct) and n_words > 1000:
        prefix_30 = ' '.join(words[:30])
        rest = ' '.join(words[30:])
        m = heading_re.search(rest)
        if m:
            cut_pos = len(prefix_30) + 1 + m.start()
            ext = txt[:cut_pos].strip()
            ext_words = len(ext.split())
            if ext_words >= 30:
                st = 'long_cut'
                rl = f'cut_before_{m.group(0).replace(" ", "_")}'
                ext_txt = ext
                ext_w = ext_words
            else:
                st = 'manual_review'
                rl = 'extract_under_30_words'
                ext_txt = txt
                ext_w = n_words
        else:
            st = 'manual_review'
            rl = 'no_heading_found_after_word_30'
            ext_txt = txt
            ext_w = n_words
    elif (not is_struct) and n_words <= 400:
        st = 'short_keep'
        rl = 'non_structural_words_le_400'
        ext_txt = txt
        ext_w = n_words
    elif (not is_struct) and (401 <= n_words <= 1000):
        st = 'medium_review'
        rl = 'non_structural_words_401_to_1000'
        ext_txt = txt
        ext_w = n_words
    else:
        st = 'manual_review'
        rl = 'structural_start_words_le_1000'
        ext_txt = txt
        ext_w = n_words
        
    status_list.append(st)
    rule_list.append(rl)
    extracted_text_list.append(ext_txt)
    extracted_words_list.append(ext_w)

df['status'] = status_list
df['rule'] = rule_list
df['extracted_text'] = extracted_text_list
df['extracted_words'] = extracted_words_list

# Save proposal CSV
proposal_cols = ['Country', 'orig_cluster', 'orig_words', 'status', 'rule', 'extracted_words', 'extracted_text']
df[proposal_cols].to_csv(proposal_csv_path, index=False)
print(f"Saved proposal CSV to: {proposal_csv_path}")
print(f"Total rows in proposal: {len(df)}")


Saved proposal CSV to: ../data/processed/preamble_extraction_proposal.csv
Total rows in proposal: 193


## 4. Step 3: Diagnostic Reports

We generate detailed reports inspecting status distributions, cut boundaries, review candidates, and short preambles:
- **3a**: Status value counts and status by original cluster crosstab.
- **3b**: `long_cut` cut boundaries (120 characters before and 120 characters after cut).
- **3c**: `medium_review` countries (first 120 and last 150 characters).
- **3d**: `manual_review` countries (first 150 and last 150 characters).
- **3e**: `no_preamble_in_source` countries (Country, orig_words, orig_cluster).
- **3f**: Preambles with extracted_words < 30.

In [5]:
print("=== REPORT 3A: STATUS VALUE COUNTS ===")
status_counts = df['status'].value_counts()
display(pd.DataFrame({'Count': status_counts, 'Percentage': (status_counts / len(df) * 100).round(2)}))

print("\n=== REPORT 3A: CROSSTAB OF STATUS BY ORIGINAL CLUSTER ===")
crosstab_df = pd.crosstab(df['status'], df['orig_cluster'], margins=True)
display(crosstab_df)


=== REPORT 3A: STATUS VALUE COUNTS ===


,Count,Percentage
status,,
short_keep,116,60.10
no_preamble_in_source,30,15.54
medium_review,29,15.03
manual_review,12,6.22
long_cut,6,3.11



=== REPORT 3A: CROSSTAB OF STATUS BY ORIGINAL CLUSTER ===


orig_cluster,0,1,2,3,4,5,6,All
status,,,,,,,,
long_cut,0,0,2,1,3,0,0,6
manual_review,0,2,5,1,2,2,0,12
medium_review,1,0,5,6,8,1,8,29
no_preamble_in_source,0,0,7,0,11,12,0,30
short_keep,25,15,6,12,17,13,28,116
All,26,17,25,20,41,28,36,193


In [6]:
print("=== REPORT 3B: LONG_CUT BOUNDARY INSPECTION ===")
long_cut_df = df[df['status'] == 'long_cut'].sort_values('Country')

for _, r in long_cut_df.iterrows():
    c = r['Country']
    txt = r['cleaned_raw']
    ext = r['extracted_text']
    cut_pos = len(ext)
    after_slice = txt[cut_pos:].lstrip()
    
    before_cut = ext[-120:] if len(ext) >= 120 else ext
    after_cut = after_slice[:120]
    
    print(f"\nCountry: {c}")
    print(f"  Original words:  {r['orig_words']}")
    print(f"  Extracted words: {r['extracted_words']}")
    print(f"  Rule applied:    {r['rule']}")
    print(f"  120 chars BEFORE cut: {repr(before_cut)}")
    print(f"  120 chars AFTER cut:  {repr(after_cut)}")


=== REPORT 3B: LONG_CUT BOUNDARY INSPECTION ===

Country: France
  Original words:  13608
  Extracted words: 102
  Rule applied:    cut_before_ARTICLE
  120 chars BEFORE cut: 'd on the common ideal of liberty, equality and fraternity and conceived for the purpose of their democratic development.'
  120 chars AFTER cut:  'ARTICLE France shall be an indivisible, secular, democratic and social Republic. It shall ensure the equality of all cit'

Country: Libya
  Original words:  2952
  Extracted words: 211
  Rule applied:    cut_before_CHAPTER_ONE
  120 chars BEFORE cut: 'ercise of power in the transitional period until the adoption of a permanent Constitution by general popular referendum.'
  120 chars AFTER cut:  'CHAPTER ONE. GENERAL PROVISIONS Article Libya shall be an independent democratic state in which the people shall be the '

Country: New Zealand
  Original words:  3310
  Extracted words: 211
  Rule applied:    cut_before_Part
  120 chars BEFORE cut: 'Parts, as follows:Division 

In [7]:
print("=== REPORT 3C: MEDIUM_REVIEW (401 TO 1000 WORDS) ===")
med_df = df[df['status'] == 'medium_review'].sort_values('Country')
print(f"Total medium_review countries: {len(med_df)}")

for _, r in med_df.iterrows():
    c = r['Country']
    txt = r['cleaned_raw']
    w = r['orig_words']
    first_120 = txt[:120]
    last_150 = txt[-150:]
    print(f"\nCountry: {c} | Words: {w} | Cluster: {r['orig_cluster']}")
    print(f"  First 120 chars: {repr(first_120)}")
    print(f"  Last 150 chars:  {repr(last_150)}")


=== REPORT 3C: MEDIUM_REVIEW (401 TO 1000 WORDS) ===
Total medium_review countries: 29

Country: Angola | Words: 580 | Cluster: 4
  First 120 chars: 'We, the people of Angola, through its lawful representatives, the legislators of the nation, freely elected in the parli'
  Last 150 chars:  'ure generations and to the exercise of our sovereignty; We hereby pass this Constitution as the Supreme and Fundamental Law of the Republic of Angola.'

Country: Bahrain | Words: 742 | Cluster: 2
  First 120 chars: 'In the name of God, the Merciful, the Compassionate Constitution of the Kingdom of Bahrain Foreword to the Constitution '
  Last 150 chars:  'al Action Charter and that complement all the unamended texts. We have attached an explanation memorandum which will be used to explain its judgement.'

Country: Belize | Words: 486 | Cluster: 4
  First 120 chars: 'WHEREAS the people of Belize-affirm that the Nation of Belize shall be founded upon principles which acknowledge the sup'
  Last 150 ch

In [8]:
print("=== REPORT 3D: MANUAL_REVIEW COUNTRIES ===")
manual_df = df[df['status'] == 'manual_review'].sort_values('Country')
print(f"Total manual_review countries: {len(manual_df)}")

for _, r in manual_df.iterrows():
    c = r['Country']
    txt = r['cleaned_raw']
    w = r['orig_words']
    first_150 = txt[:150]
    last_150 = txt[-150:]
    print(f"\nCountry: {c} | Words: {w} | Cluster: {r['orig_cluster']} | Rule: {r['rule']}")
    print(f"  First 150 chars: {repr(first_150)}")
    print(f"  Last 150 chars:  {repr(last_150)}")


=== REPORT 3D: MANUAL_REVIEW COUNTRIES ===
Total manual_review countries: 12

Country: Algeria | Words: 1307 | Cluster: 2 | Rule: no_heading_found_after_word_30
  First 150 chars: 'The Algerian people are a free people; and they are resolved to remain so. Stretching back over thousands of years, their history is marked by a progr'
  Last 150 chars:  'uture generations, the successors of the pioneers of freedom and the founders of the free society. This shall be part and parcel of this Constitution.'

Country: Armenia | Words: 78 | Cluster: 1 | Rule: structural_start_words_le_1000
  First 150 chars: 'The Armenian People, accepting as a basis the fundamental principles of Armenian statehood and pan-national aspirations enshrined in the Declaration o'
  Last 150 chars:  '-being, and civic solidarity of the generations, and affirming its commitment to universal values, adopts the Constitution of the Republic of Armenia.'

Country: China | Words: 1160 | Cluster: 3 | Rule: no_heading_found_a

In [9]:
print("=== REPORT 3E: NO_PREAMBLE_IN_SOURCE COUNTRIES ===")
no_preamble_df = df[df['status'] == 'no_preamble_in_source'][['Country', 'orig_words', 'orig_cluster']].sort_values('Country')
print(f"Total no_preamble_in_source countries: {len(no_preamble_df)}")
display(no_preamble_df)


=== REPORT 3E: NO_PREAMBLE_IN_SOURCE COUNTRIES ===
Total no_preamble_in_source countries: 30


,Country,orig_words,orig_cluster
8,Australia,16303,4
9,Austria,47164,5
22,Botswana,30727,4
30,Canada,20145,4
44,Cyprus,34466,5
46,Denmark,6023,5
58,Finland,12987,5
80,Israel,17607,2
81,Italy,11305,5
82,Jamaica,36267,4


In [10]:
print("=== REPORT 3F: EXTRACTED WORDS < 30 ===")
short_extracted_df = df[df['extracted_words'] < 30].sort_values('extracted_words', ascending=False)

non_empty_short = short_extracted_df[short_extracted_df['extracted_words'] > 0]
print(f"1. Short Preambles with Non-Empty Text (0 < words < 30): {len(non_empty_short)} countries")
for _, r in non_empty_short.iterrows():
    print(f"\nCountry: {r['Country']} | Words: {r['extracted_words']} | Status: {r['status']} | Cluster: {r['orig_cluster']}")
    print(f"  Full Text: {repr(r['extracted_text'])}")

empty_short = short_extracted_df[short_extracted_df['extracted_words'] == 0]
print(f"\n2. Constitutions with No Preamble in Source (words = 0, empty text): {len(empty_short)} countries")
print(f"Countries: {', '.join(empty_short['Country'].tolist())}")


=== REPORT 3F: EXTRACTED WORDS < 30 ===
1. Short Preambles with Non-Empty Text (0 < words < 30): 3 countries

Country: Latvia | Words: 15 | Status: short_keep | Cluster: 1
  Full Text: 'The people of Latvia, in freely elected Constitutional Assembly, have adopted the following State Constitution:'

Country: Greece | Words: 11 | Status: short_keep | Cluster: 0
  Full Text: 'In the name of the Holy and Consubstantial and Indivisible Trinity'

Country: Belgium | Words: 11 | Status: short_keep | Cluster: 5
  Full Text: 'In the name of the Belgian people, the National Congress decrees:'

2. Constitutions with No Preamble in Source (words = 0, empty text): 30 countries
Countries: Australia, Botswana, Canada, Cyprus, Denmark, Austria, Finland, Israel, Italy, Jamaica, Lesotho, Luxembourg, Malaysia, Maldives, Malta, Mauritius, Mexico, Monaco, Netherlands, Norway, Oman, Qatar, Romania, Saudi Arabia, Sierra Leone, Singapore, Somalia, Sweden, Uruguay, Yemen


# Section 5: Task 3b Refinement & Decision Framework

This section refines the preamble extraction proposal based on targeted diagnostic audits:
1. **Part A: Fixed Heading Detector**: Implements a case-sensitive, context-aware heading detector (`CHAPTER, Chapter, PART, Part, TITLE, Title, ARTICLE, Article, Art, SECTION, Section, BOOK, Book`), requiring strict punctuation or text start boundaries and eliminating false positives (e.g. "inseparable part"). "Constitution Act" is removed from the structural keyword list.
2. **Part B: Canada Discrepancy Analysis**: Audits the Canadian constitutional source text to account for word count differences between the 1867 British North America Act, the 1982 Charter of Rights and Freedoms, and the consolidated text.
3. **Part C: No-Preamble In-Source Audit**: Conducts a case-sensitive cue audit across all 31 countries without preambles (including New Zealand), and isolates Australia's covering clause preamble.
4. **Part D: Decision Implementation**: Applies explicit classifications across all edge cases (`long_cut` acceptance, table of contents reclassification, long preamble approvals, Croatian heading strip, Lebanese cut proposal, formula-only isolations, and medium-review heading cuts).
5. **Part E: V2 Dataset & Summary Audits**: Exports `data/processed/preamble_extraction_v2.csv` (with `notes`) and computes final status, clusterable set statistics, and cluster crosstabs.

## 5.1 Part A: Fixed Heading Detector & Status Impact

A real heading match must satisfy four conditions:
1. Keyword must be one of: `CHAPTER`, `Chapter`, `PART`, `Part`, `TITLE`, `Title`, `ARTICLE`, `Article`, `Art`, `SECTION`, `Section`, `BOOK`, `Book` (case-sensitive).
2. Followed by a numeral, Roman numeral, ordinal word, or capitalised word.
3. Located at the start of text, after sentence-ending punctuation (`[.!?]`, `[:]`), or after another heading.
4. Preceding words must NOT be lowercase (e.g., "inseparable part", "which is part of", "outrageous article" are excluded).
5. "Constitution Act" is removed from the keyword list.

We re-run the status assignment with this fixed detector and report all countries whose status changed compared to the proposal.

In [11]:
kw = r'(?:CHAPTER|Chapter|PART|Part|TITLE|Title|ARTICLE|Article|Art|SECTION|Section|BOOK|Book)'
target = r'(?:\d+|[IVXLCDM]+|[A-Z][a-zA-Z]*|first|second|third|fourth|fifth|One|Two|Three|Four|Five|Six|Seven|Eight|Nine|Ten)'

heading_re_fixed = re.compile(
    r'(?:^|[.:;!?]\s+|(?:' + kw + r'\s+' + target + r'[\s.:]+))' + 
    r'(' + kw + r'\s+' + target + r')\b'
)

extended_phrases_new = [
    'fundamental principles', 'basic law', 'general provisions', 'instrument of government'
]

def check_structure_start_fixed(text):
    words = text.split()
    first_12 = words[:12]
    first_12_str = ' '.join(first_12)
    has_phrase = any(phrase in first_12_str.lower() for phrase in extended_phrases_new)
    m = heading_re_fixed.search(first_12_str)
    has_heading = bool(m)
    has_roman_start = bool(re.match(r'^[IVXLCDM]+\s+[A-Z]', first_12_str))
    return has_phrase or has_heading or has_roman_start

status_list_new = []
rule_list_new = []
extracted_text_list_new = []
extracted_words_list_new = []

for i, r in df.iterrows():
    txt = r['cleaned_raw']
    words = txt.split()
    n_words = len(words)
    is_struct = check_structure_start_fixed(txt)
    
    if is_struct and n_words > 1000:
        st = 'no_preamble_in_source'
        rl = 'structural_start_and_words_gt_1000'
        ext_txt = ''
        ext_w = 0
    elif (not is_struct) and n_words > 1000:
        prefix_30 = ' '.join(words[:30])
        rest = txt[len(prefix_30):]
        m = heading_re_fixed.search(rest)
        if m:
            cut_pos = len(prefix_30) + m.start(1)
            ext = txt[:cut_pos].strip()
            ext_words = len(ext.split())
            if ext_words >= 30:
                st = 'long_cut'
                rl = f'cut_before_{m.group(1).replace(" ", "_")}'
                ext_txt = ext
                ext_w = ext_words
            else:
                st = 'manual_review'
                rl = 'extract_under_30_words'
                ext_txt = txt
                ext_w = n_words
        else:
            st = 'manual_review'
            rl = 'no_heading_found_after_word_30'
            ext_txt = txt
            ext_w = n_words
    elif (not is_struct) and n_words <= 400:
        st = 'short_keep'
        rl = 'non_structural_words_le_400'
        ext_txt = txt
        ext_w = n_words
    elif (not is_struct) and (401 <= n_words <= 1000):
        st = 'medium_review'
        rl = 'non_structural_words_401_to_1000'
        ext_txt = txt
        ext_w = n_words
    else:
        st = 'manual_review'
        rl = 'structural_start_words_le_1000'
        ext_txt = txt
        ext_w = n_words
        
    status_list_new.append(st)
    rule_list_new.append(rl)
    extracted_text_list_new.append(ext_txt)
    extracted_words_list_new.append(ext_w)

df['status_detector_rerun'] = status_list_new

df_prop = pd.read_csv(proposal_csv_path)
df['status_proposal'] = df_prop['status']

changed_mask = df['status_detector_rerun'] != df['status_proposal']
df_changes = df[changed_mask].copy()

print(f"=== STATUS CHANGES AFTER FIXING HEADING DETECTOR ({len(df_changes)} COUNTRIES) ===\n")

reasons = {
    'Australia': 'Removed "Constitution Act" from structural start keywords; text starts with imperial Act title before preamble at word 13.',
    'Canada': 'Removed "Constitution Act" from structural start keywords; text starts with Act title before 1867 preamble at word 27.',
    'Liechtenstein': 'Eliminated false positive Roman numeral "II" in regnal name "John II"; genuine 41-word promulgation formula.',
    'Norway': 'First 12 words ("A Form of government and religion Article The") lacks standard punctuation before "Article".',
    'Syria': 'Eliminated false positive "part" in prose clause "which is part of human heritage"; 548-word text is non-structural start.',
    'Tajikistan': 'Eliminated false positive "part" in prose clause "As an inseparable part of the world community"; 70-word text is non-structural start.',
    'Tonga': 'Eliminated false positive Roman numeral "I" in regnal name "King George Tupou I"; 129-word text is non-structural start.'
}

change_records = []
for _, r in df_changes.iterrows():
    c = r['Country']
    change_records.append({
        'Country': c,
        'Old Status (Proposal)': r['status_proposal'],
        'New Status (Fixed Detector)': r['status_detector_rerun'],
        'Words': r['orig_words'],
        'Reason': reasons.get(c, 'Heuristic refinement')
    })

display(pd.DataFrame(change_records))


=== STATUS CHANGES AFTER FIXING HEADING DETECTOR (7 COUNTRIES) ===



,Country,Old Status (Proposal),New Status (Fixed Detector),Words,Reason
0,Australia,no_preamble_in_source,long_cut,16303,"Removed ""Constitution Act"" from structural start keywords; text starts with imperial Act title before preamble at word 13."
1,Canada,no_preamble_in_source,long_cut,20145,"Removed ""Constitution Act"" from structural start keywords; text starts with Act title before 1867 preamble at word 27."
2,Liechtenstein,manual_review,short_keep,41,"Eliminated false positive Roman numeral ""II"" in regnal name ""John II""; genuine 41-word promulgation formula."
3,Norway,no_preamble_in_source,long_cut,7231,"First 12 words (""A Form of government and religion Article The"") lacks standard punctuation before ""Article""."
4,Syria,manual_review,medium_review,548,"Eliminated false positive ""part"" in prose clause ""which is part of human heritage""; 548-word text is non-structural start."
5,Tajikistan,manual_review,short_keep,70,"Eliminated false positive ""part"" in prose clause ""As an inseparable part of the world community""; 70-word text is non-structural start."
6,Tonga,manual_review,short_keep,129,"Eliminated false positive Roman numeral ""I"" in regnal name ""King George Tupou I""; 129-word text is non-structural start."


## 5.2 Part B: Canada Discrepancy Investigation

We investigate the discrepancy regarding Canada's word count across sources and iterations:
- Printing row index and duplicate row check.
- Word count and first/last 200 characters from `data/processed/constitution_dataset_cleaned.csv` and `data/raw/constitution_dataset.csv`.
- Explaining the origin of each count and text segment.

In [12]:
print("=== CANADA DISCREPANCY AUDIT ===\n")

clean_canada_rows = df_clean[df_clean['Country'] == 'Canada']
print(f"Processed cleaned CSV - Row index: {clean_canada_rows.index.tolist()}")
print(f"Duplicate rows in processed cleaned CSV: {len(clean_canada_rows) > 1}")

df_raw_source = pd.read_csv('../data/raw/constitution_dataset.csv')
raw_canada_rows = df_raw_source[df_raw_source['Country'] == 'Canada']
print(f"Raw source CSV - Row index: {raw_canada_rows.index.tolist()}")
print(f"Duplicate rows in raw source CSV: {len(raw_canada_rows) > 1}")

clean_can_text = clean_canada_rows.iloc[0]['Preamble']
clean_can_words = len(str(clean_can_text).split())
print(f"\nProcessed Cleaned Text Word Count: {clean_can_words}")
print(f"First 200 chars:\n  {repr(clean_can_text[:200])}")
print(f"Last 200 chars:\n  {repr(clean_can_text[-200:])}")

raw_can_text = raw_canada_rows.iloc[0]['Preamble']
raw_can_words = len(str(raw_can_text).split())
print(f"\nRaw Source Dataset Word Count: {raw_can_words}")
print(f"First 200 chars:\n  {repr(str(raw_can_text)[:200])}")
print(f"Last 200 chars:\n  {repr(str(raw_can_text)[-200:])}")

idx_1867 = clean_can_text.find('WHEREAS the Provinces of Canada')
idx_1867_end = clean_can_text.find('1. Short title', idx_1867)
preamble_1867 = clean_can_text[idx_1867:idx_1867_end].strip()
print(f"\n1867 Constitution Act Preamble: {len(preamble_1867.split())} words (chars {idx_1867} to {idx_1867_end})")
print(f"Snippet: {repr(preamble_1867[:150])}...")

idx_1982 = clean_can_text.find('Whereas Canada is founded upon principles')
idx_1982_end = clean_can_text.find('A. GUARANTEE OF RIGHTS', idx_1982)
preamble_1982 = clean_can_text[idx_1982:idx_1982_end].strip()
print(f"1982 Charter of Rights Preamble: {len(preamble_1982.split())} words (at char {idx_1982})")
print(f"Text: {repr(preamble_1982)}")


=== CANADA DISCREPANCY AUDIT ===

Processed cleaned CSV - Row index: [30]
Duplicate rows in processed cleaned CSV: False


Raw source CSV - Row index: [30]
Duplicate rows in raw source CSV: False

Processed Cleaned Text Word Count: 20145
First 200 chars:
  'CONSTITUTION ACT An Act for the Union of Canada, Nova Scotia, and New Brunswick, and the Government thereof; and for Purposes connected therewith 29th March  WHEREAS the Provinces of Canada, Nova Scot'
Last 200 chars:
  '- Eliz. II, c. Can. Column II. Amendment Section is repealed and the following substituted therefor: . This Act may be cited as the Constitution Act No. , . Column III. New Name Constitution Act No. ,'

Raw Source Dataset Word Count: 5434
First 200 chars:
  'CONSTITUTION ACT 1867 JUDICATURE; THE FIRST SCHEDULE. ELECTORAL DISTRICTS OF ONTARIO CONSTITUTION ACT 1867An Act for the Union of Canada, Nova Scotia, and New Brunswick, and the Government thereof; an'
Last 200 chars:
  'esignation, Disqualification, etcThe Place of a Legislative Councillor of Quebec shall become vacant in the Cases, mutatis mutandis, in which the Place of Senator b

## 5.3 Part C: No-Preamble In-Source Audit

For every country classified as `no_preamble_in_source` in the proposal (30 countries) plus New Zealand (31 countries total):
1. We print the first 150 characters.
2. We perform a case-sensitive search across the entire text for preamble indicator cues: `"Preamble"`, `"PREAMBLE"`, `"Foreword"`, `"We, the people"`, `"We the People"`, `"WE, THE PEOPLE"`, `"Whereas"`, `"WHEREAS"`, `"In the name of"`.
3. Up to 2 earliest hits are reported with character offset and 200 characters of context.
4. For Australia, we isolate and print the text from `"WHEREAS"` up to the next heading or `"BE IT"`.

In [13]:
print("=== NO-PREAMBLE IN-SOURCE AUDIT (31 COUNTRIES) ===\n")

no_preamble_countries = df_prop[df_prop['status'] == 'no_preamble_in_source']['Country'].tolist()
if 'New Zealand' not in no_preamble_countries:
    no_preamble_countries.append('New Zealand')
no_preamble_countries = sorted(no_preamble_countries)

cues = ['Preamble', 'PREAMBLE', 'Foreword', 'We, the people', 'We the People', 'WE, THE PEOPLE', 'Whereas', 'WHEREAS', 'In the name of']

for c in no_preamble_countries:
    txt = str(df.loc[df['Country'] == c, 'cleaned_raw'].values[0])
    w_count = len(txt.split())
    print(f"Country: {c} (Total words: {w_count:,}, chars: {len(txt):,})")
    print(f"  First 150 chars: {repr(txt[:150])}")
    
    hits = []
    for cue in cues:
        start_pos = 0
        while True:
            pos = txt.find(cue, start_pos)
            if pos == -1:
                break
            hits.append((pos, cue))
            start_pos = pos + len(cue)
            
    hits.sort(key=lambda x: x[0])
    if hits:
        print(f"  Cue hits found ({len(hits)} total):")
        for pos, cue in hits[:2]:
            ctx_start = max(0, pos - 30)
            ctx_end = min(len(txt), pos + len(cue) + 170)
            print(f"    [{cue}] at char {pos:,}:")
            print(f"      Context: {repr(txt[ctx_start:ctx_end])}")
    else:
        print("  No preamble cues found.")
    print()

aus_txt = str(df.loc[df['Country'] == 'Australia', 'cleaned_raw'].values[0])
idx_w = aus_txt.find('WHEREAS')
idx_beit = aus_txt.find('Be it therefore enacted', idx_w)
if idx_beit == -1:
    idx_beit = aus_txt.find('BE IT', idx_w)

print("=== AUSTRALIA: 'WHEREAS' UP TO NEXT HEADING / 'BE IT' ===")
aus_preamble = aus_txt[idx_w:idx_beit].strip()
print(f"Word count: {len(aus_preamble.split())} words")
print(f"Text:\n{aus_preamble}")


=== NO-PREAMBLE IN-SOURCE AUDIT (31 COUNTRIES) ===

Country: Australia (Total words: 16,303, chars: 96,045)
  First 150 chars: 'Commonwealth of Australia Constitution Act An Act to constitute the Commonwealth of Australia WHEREAS the people of New South Wales, Victoria, South A'
  Cue hits found (10 total):
    [WHEREAS] at char 94:
      Context: 'the Commonwealth of Australia WHEREAS the people of New South Wales, Victoria, South Australia, Queensland, and Tasmania, humbly relying on the blessing of Almighty God, have agreed to unite in one indissolu'
    [WHEREAS] at char 73,956:
      Context: 'Majesty the King and Germany. WHEREAS certain legal difficulties exist which have created doubts and caused delays in relation to certain Commonwealth legislation, and to certain regulations made thereunder,'

Country: Austria (Total words: 47,164, chars: 290,996)
  First 150 chars: 'Chapter I General Provisions. European Union A. General Provisions Art Austria is a democratic republic. Its 

Country: Sierra Leone (Total words: 43,149, chars: 257,711)
  First 150 chars: 'CHAPTER I THE REPUBLIC OF SIERRA LEONE . Declaration of Republic Sierra Leone is a Sovereign Republic, the boundaries of which are delimited in the Fi'
  Cue hits found (1 total):
    [Whereas] at char 9,657:
      Context: 'nd Freedoms of the Individual Whereas every person in Sierra Leone is entitled to the fundamental human rights and freedoms of the individual, that is to say, has the right, whatever his race, tribe, place o'

Country: Singapore (Total words: 46,473, chars: 275,662)
  First 150 chars: 'PART I PRELIMINARY . Citation This Constitution may be cited as the Constitution of the Republic of Singapore. . Interpretation In this Constitution, '
  No preamble cues found.



Country: Somalia (Total words: 20,776, chars: 129,362)


  First 150 chars: 'CHAPTER DECLARATION OF THE FEDERAL REPUBLIC OF SOMALIA Article . The Federal Republic of Somalia Somalia is a federal, sovereign, and democratic repub'
  No preamble cues found.

Country: Sweden (Total words: 58,431, chars: 352,078)
  First 150 chars: 'The Instrument of Government Chapter . Basic principles of the form of government Art All public power in Sweden proceeds from the people. Swedish dem'
  No preamble cues found.

Country: Uruguay (Total words: 29,544, chars: 183,968)
  First 150 chars: 'SECTION I The Nation and Its Sovereignty Chapter I Article The Oriental Republic of Uruguay is the political association of all inhabitants included w'
  No preamble cues found.

Country: Yemen (Total words: 11,615, chars: 72,765)
  First 150 chars: 'PART ONE THE FOUNDATIONS OF STATE Chapter I. The Political Foundations Article The Republic of Yemen is an Arab, Islamic and independent sovereign sta'
  No preamble cues found.

=== AUSTRALIA: 'WHEREAS' UP TO NEXT HEADIN

## 5.4 Part D: Decision Implementation & V2 Dataset Generation

We implement the 7 specific decisions mandated for Task 3b and record each decision in the `notes` column:
1. **Accept `long_cut`**: France, Libya, Palestine, Slovakia, Suriname -> `preamble_cut`.
2. **New Zealand**: Set to `no_preamble_in_source` (source text was a table of contents).
3. **Accept `preamble_ok`**: Algeria, Armenia, China, Egypt, Iran, Papua New Guinea, Syria, Tajikistan.
4. **Croatia**: Strip duplicated leading heading `"I Historical Foundations I Historical Foundations"`. Verify that body articles do not bleed into the text.
5. **Lebanon**: Print words 150 to 450. Identify cut point before `Chapter One` / `Article 1` at character offset ~1,790 (word 272). Retain in `manual_review` per instructions (do not apply cut yet).
6. **`formula_only`**: Belgium, Greece, Latvia, Liechtenstein, Tonga. Audit any other country with fewer than 50 extracted words.
7. **`medium_review`**: Apply real heading cut rule after word 30. Cuba drops trailing `"CONSTITUTION"`. Preambles without headings remain intact. Print Bahrain's first 400 characters.

The output is written to `data/processed/preamble_extraction_v2.csv`.

In [14]:
cro_raw = df.loc[df['Country'] == 'Croatia', 'cleaned_raw'].values[0]
dup_head = 'I Historical Foundations I Historical Foundations'
cro_stripped = cro_raw.replace(dup_head, '').strip()
print("=== CROATIA: LAST 300 CHARACTERS INSPECTION ===")
print(repr(cro_stripped[-300:]))
print("Does body text bleed in? No: the text concludes at the end of the historical preamble;")
print('Article 1 ("The Republic of Croatia is a unitary and indivisible...')
print("is not present in this record.\n")

leb_raw = df.loc[df['Country'] == 'Lebanon', 'cleaned_raw'].values[0]
leb_words = leb_raw.split()
print("=== LEBANON: WORDS 150 TO 450 ===")
print(' '.join(leb_words[150:450]))

leb_target = 'charter of co-existence.'
leb_cut_pos = leb_raw.find(leb_target) + len(leb_target)
print("\n=== LEBANON: PROPOSED CUT POINT ===")
print(f"Proposed cut position: char {leb_cut_pos} (word {len(leb_raw[:leb_cut_pos].split())})")
print(f"150 chars BEFORE cut: {repr(leb_raw[leb_cut_pos-150:leb_cut_pos])}")
print(f"150 chars AFTER cut:  {repr(leb_raw[leb_cut_pos:leb_cut_pos+150])}\n")

bahrain_raw = df.loc[df['Country'] == 'Bahrain', 'cleaned_raw'].values[0]
print("=== BAHRAIN: FIRST 400 CHARACTERS ===")
print(repr(bahrain_raw[:400]))
print()

v2_status = []
v2_rule = []
v2_ext_text = []
v2_ext_words = []
v2_notes = []

formula_countries = {'Belgium', 'Greece', 'Latvia', 'Liechtenstein', 'Tonga'}
accepted_long_cut = {'France', 'Libya', 'Palestine', 'Slovakia', 'Suriname'}
accepted_preamble_ok = {'Algeria', 'Armenia', 'China', 'Egypt', 'Iran', 'Papua New Guinea', 'Syria', 'Tajikistan'}

for idx, r in df.iterrows():
    c = r['Country']
    txt = r['cleaned_raw']
    w = r['orig_words']
    old_st = df_prop.loc[idx, 'status']
    old_ext = df_prop.loc[idx, 'extracted_text']
    old_ew = df_prop.loc[idx, 'extracted_words']
    
    if c == 'New Zealand':
        st = 'no_preamble_in_source'
        rl = 'table_of_contents_only'
        ext = ''
        ew = 0
        notes = 'table of contents only; no preamble in source'
    elif c in formula_countries:
        st = 'formula_only'
        rl = 'formula_promulgation_only'
        ext = txt
        ew = w
        notes = 'promulgation formula only (< 50 words or ceremonial); excluded from clustering'
    elif c in accepted_long_cut:
        st = 'preamble_cut'
        rl = df_prop.loc[idx, 'rule']
        ext = str(old_ext)
        ew = int(old_ew)
        notes = 'accepted cut before first body heading'
    elif c == 'Croatia':
        st = 'preamble_ok'
        rl = 'stripped_duplicate_heading'
        ext = cro_stripped
        ew = len(cro_stripped.split())
        notes = 'stripped duplicated leading heading; genuine preamble confirmed (no body article bleed-in)'
    elif c == 'Lebanon':
        st = 'manual_review'
        rl = 'pending_cut_proposal'
        ext = txt
        ew = w
        notes = 'proposed cut before Chapter One / Article 1 at char ~1790 (word 272); held in manual_review per prompt'
    elif c in accepted_preamble_ok:
        st = 'preamble_ok'
        rl = 'accepted_full_preamble'
        ext = txt
        ew = w
        notes = 'accepted as genuine full-length preamble'
    elif c == 'Cuba':
        trailing = 'CONSTITUTION'
        ext = txt
        if ext.endswith(trailing):
            ext = ext[:-len(trailing)].strip()
        if ext.endswith(':'):
            ext = ext[:-1].strip()
        st = 'preamble_cut'
        rl = 'drop_trailing_constitution'
        ew = len(ext.split())
        notes = 'dropped trailing label CONSTITUTION'
    elif old_st == 'medium_review':
        st = 'preamble_ok'
        rl = 'non_structural_words_401_to_1000'
        ext = txt
        ew = w
        notes = 'medium-length preamble; no body headings found, kept intact'
    elif old_st == 'short_keep':
        st = 'preamble_ok'
        rl = 'non_structural_words_le_400'
        ext = txt
        ew = w
        notes = 'standard clean preamble (<= 400 words)'
    elif old_st == 'no_preamble_in_source':
        st = 'no_preamble_in_source'
        rl = 'structural_start_and_words_gt_1000'
        ext = ''
        ew = 0
        notes = 'no preamble in source; full constitution body'
    else:
        st = old_st
        rl = df_prop.loc[idx, 'rule']
        ext = txt
        ew = w
        notes = 'retained'
        
    v2_status.append(st)
    v2_rule.append(rl)
    v2_ext_text.append(ext)
    v2_ext_words.append(ew)
    v2_notes.append(notes)

df_v2 = df[['Country', 'orig_cluster', 'orig_words']].copy()
df_v2['status'] = v2_status
df_v2['rule'] = v2_rule
df_v2['extracted_words'] = v2_ext_words
df_v2['extracted_text'] = v2_ext_text
df_v2['notes'] = v2_notes

v2_csv_path = os.path.join(data_dir, 'preamble_extraction_v2.csv')
df_v2.to_csv(v2_csv_path, index=False)
print(f"Saved V2 extraction dataset to: {v2_csv_path}")
print(f"Total rows: {len(df_v2)}")

under_50_df = df_v2[(df_v2['extracted_words'] > 0) & (df_v2['extracted_words'] < 50)].sort_values('extracted_words')
print(f"\n=== ALL COUNTRIES WITH EXTRACTED WORDS < 50 ({len(under_50_df)}) ===")
display(under_50_df[['Country', 'status', 'extracted_words', 'notes']])


=== CROATIA: LAST 300 CHARACTERS INSPECTION ===
'and all citizens, resolutely expressed in the free elections, the Republic of Croatia is hereby founded and shall develop as a sovereign and democratic state in which equality, freedoms and human rights are guaranteed and ensured, and their economic and cultural progress and social welfare promoted.'
Does body text bleed in? No: the text concludes at the end of the historical preamble;
Article 1 ("The Republic of Croatia is a unitary and indivisible...
is not present in this record.

=== LEBANON: WORDS 150 TO 450 ===
through constitutional institutions. The system is based upon the principle of separation of powers, their balance, and their cooperation. The economic system is free, guaranteeing individual initiative and private ownership. Balanced development of regions, culturally, socially, and economically, is a cornerstone of the State unity and the system stability. Eliminating political sectarianism is a basic national objective, t

,Country,status,extracted_words,notes
16,Belgium,formula_only,11,promulgation formula only (< 50 words or ceremonial); excluded from clustering
65,Greece,formula_only,11,promulgation formula only (< 50 words or ceremonial); excluded from clustering
92,Latvia,formula_only,15,promulgation formula only (< 50 words or ceremonial); excluded from clustering
40,Costa Rica,preamble_ok,33,standard clean preamble (<= 400 words)
133,Peru,preamble_ok,33,standard clean preamble (<= 400 words)
96,Liechtenstein,formula_only,41,promulgation formula only (< 50 words or ceremonial); excluded from clustering
84,Jordan,preamble_ok,42,standard clean preamble (<= 400 words)
130,Panama,preamble_ok,43,standard clean preamble (<= 400 words)


## 5.5 Part E: Summary Statistics & Final Audits

We present the final summary of the refined extraction proposal:
1. Status counts and distribution across all 193 countries.
2. Number of clusterable preambles (`preamble_ok` + `preamble_cut`).
3. Word count distribution (min, median, mean, max) and count exceeding 300 words.
4. Crosstabulation of final status against original cluster assignments.

In [15]:
print("=== FINAL V2 STATUS VALUE COUNTS ===")
status_counts_v2 = df_v2['status'].value_counts()
display(pd.DataFrame({'Count': status_counts_v2, 'Percentage': (status_counts_v2 / len(df_v2) * 100).round(2)}))

clusterable_mask = df_v2['status'].isin(['preamble_ok', 'preamble_cut'])
df_clusterable = df_v2[clusterable_mask]
c_words = df_clusterable['extracted_words']

print(f"\nTotal Clusterable Preambles: {len(df_clusterable)} countries")
print(f"Excluded from clustering: {len(df_v2) - len(df_clusterable)} (31 no_preamble_in_source + 5 formula_only + 1 manual_review)")

print("\n=== CLUSTERABLE SET WORD COUNT STATISTICS ===")
stats_df = pd.DataFrame({
    'Statistic': ['Count', 'Min Words', 'Median Words', 'Mean Words', 'Max Words', 'Exceeding 300 Words (>300)'],
    'Value': [len(df_clusterable), c_words.min(), c_words.median(), round(c_words.mean(), 1), c_words.max(), int((c_words > 300).sum())]
})
display(stats_df)

print("\n=== CROSSTAB OF FINAL V2 STATUS BY ORIGINAL CLUSTER ===")
crosstab_v2 = pd.crosstab(df_v2['status'], df_v2['orig_cluster'], margins=True)
display(crosstab_v2)


=== FINAL V2 STATUS VALUE COUNTS ===


,Count,Percentage
status,,
preamble_ok,150,77.72
no_preamble_in_source,31,16.06
preamble_cut,6,3.11
formula_only,5,2.59
manual_review,1,0.52



Total Clusterable Preambles: 156 countries
Excluded from clustering: 37 (31 no_preamble_in_source + 5 formula_only + 1 manual_review)

=== CLUSTERABLE SET WORD COUNT STATISTICS ===


,Statistic,Value
0,Count,156.0
1,Min Words,33.0
2,Median Words,203.0
3,Mean Words,299.5
4,Max Words,3223.0
5,Exceeding 300 Words (>300),50.0



=== CROSSTAB OF FINAL V2 STATUS BY ORIGINAL CLUSTER ===


orig_cluster,0,1,2,3,4,5,6,All
status,,,,,,,,
formula_only,1,1,0,0,1,2,0,5
manual_review,0,0,1,0,0,0,0,1
no_preamble_in_source,0,0,7,0,12,12,0,31
preamble_cut,1,0,2,1,2,0,0,6
preamble_ok,24,16,15,19,26,14,36,150
All,26,17,25,20,41,28,36,193


# Section 6: Task 3c Final Extraction & Dataset V3

This section implements the final corrections, validation audits, and provenance tracking:
1. **Part A: Evidence Check on Medium Countries**: Audits the exact source text, word count, last 250 characters, and heading scan results for Cameroon, Chad, Morocco, Myanmar, Senegal, Thailand, Timor-Leste, and Tunisia.
2. **Part B: Comprehensive Heading Scan across all Preamble Texts**: Tests all `preamble_ok` texts for isolated or glued structural headings after word 30.
3. **Part C: Regression Fixes & Deep Audits**:
   - Re-evaluates Norway with a 15-word capitalization structural rule, returning it to the excluded group, while confirming non-structural classification for Tajikistan, Tonga, and Liechtenstein.
   - Extracts Australia's 70-word covering clause preamble.
   - Extracts and joins Canada's 1867 British North America Act preamble and 1982 Charter of Rights preamble.
   - Resolves the Canada audit discrepancy and verifies word count parity between `doc_type_audit.csv` and `constitution_dataset_cleaned.csv` across all 193 countries.
4. **Part D: Final Extraction Decisions**:
   - Applies the proposed cut to Lebanon and strips its leading duplicated heading.
   - Reclassifies Jordan as `formula_only`.
   - Retains Costa Rica, Peru, and Panama as `preamble_ok`, prints their full text, and adds the `short_flag` column.
   - Strips lone leading `"Preamble"` / `"Foreword"` labels.
   - Renames `no_preamble_in_source` to `no_preamble_in_dataset`.
   - Exports `data/processed/preamble_extraction_v3.csv`.
5. **Part E: Notebook 01 Provenance Audit**: Catalogs every hand-pasted constitution in `01_data_preprocessing.ipynb`.
6. **Part F: Final Summary Statistics & Crosstab**: Computes final status distribution, clusterable set statistics, and cluster crosstabs.

## 6.1 Part A: Evidence Check on Medium Countries

We inspect the 8 countries in the medium group (Cameroon, Chad, Morocco, Myanmar, Senegal, Thailand, Timor-Leste, and Tunisia):
- File and column source used.
- Word count.
- The last 250 characters as stored in `preamble_extraction_v2.csv`.
- First heading match position after word 30 using the fixed detector (or "none").

In [16]:
print("=== EVIDENCE CHECK ON 8 MEDIUM COUNTRIES ===\n")

df_v2 = pd.read_csv('../data/processed/preamble_extraction_v2.csv')

medium_check_countries = ['Cameroon', 'Chad', 'Morocco', 'Myanmar', 'Senegal', 'Thailand', 'Timor-Leste', 'Tunisia']

kw = r'(?:CHAPTER|Chapter|PART|Part|TITLE|Title|ARTICLE|Article|Art|SECTION|Section|BOOK|Book)'
target = r'(?:\d+|[IVXLCDM]+|[A-Z][a-zA-Z]*|first|second|third|fourth|fifth|One|Two|Three|Four|Five|Six|Seven|Eight|Nine|Ten)'
p_detector = re.compile(
    r'(?:^|[.:;!?]\s+|(?:' + kw + r'\s+' + target + r'[\s.:]+))(' + kw + r'\s+' + target + r')\b'
)

for c in medium_check_countries:
    row = df_v2[df_v2['Country'] == c].iloc[0]
    txt = str(row['extracted_text'])
    words = txt.split()
    w_count = len(words)
    last_250 = txt[-250:]
    
    # Run detector after word 30
    prefix_30 = ' '.join(words[:30])
    rest = txt[len(prefix_30):]
    m = p_detector.search(rest)
    
    match_pos_str = f"char {len(prefix_30) + m.start(1)}" if m else "none"
    
    print(f"Country: {c}")
    print(f"  Source: preamble_extraction_v2.csv ['extracted_text'] (from constitution_dataset_cleaned.csv ['Preamble'])")
    print(f"  Word count: {w_count}")
    print(f"  First heading match after word 30: {match_pos_str}")
    print(f"  Last 250 characters:\n    {repr(last_250)}\n")


=== EVIDENCE CHECK ON 8 MEDIUM COUNTRIES ===

Country: Cameroon
  Source: preamble_extraction_v2.csv ['extracted_text'] (from constitution_dataset_cleaned.csv ['Preamble'])
  Word count: 835
  First heading match after word 30: none
  Last 250 characters:
    ' the burden of public expenditure according to his financial resources; all citizens shall contribute to the defence of the Fatherland; the State shall guarantee all citizens of either sex the rights and freedoms set forth in the of the Constitution.'

Country: Chad
  Source: preamble_extraction_v2.csv ['extracted_text'] (from constitution_dataset_cleaned.csv ['Preamble'])
  Word count: 456
  First heading match after word 30: none
  Last 250 characters:
    'r commitment to the cause of African unity and our commitment to do everything possible to realize sub-regional and regional integration; Solemnly adopt this Constitution as supreme law of the State. This is made an integral part of the Constitution.'

Country: Morocco
  Sou

## 6.2 Part B: Heading Scan on Every Clusterable Text

We execute a comprehensive scan on ALL `preamble_ok` texts (after word 30) for:
1. Standard real headings (`CHAPTER`, `Chapter`, `PART`, `Part`, `TITLE`, `Title`, `ARTICLE`, `Article`, `Art`, `SECTION`, `Section`, `BOOK`, `Book` followed by numeral, Roman numeral, ordinal word, or capitalised word).
2. Glued headings (e.g. `TerritoryArticle 1` where a heading is conjoined to the preceding word without a space).

In [17]:
print("=== HEADING SCAN ACROSS ALL PREAMBLE_OK TEXTS ===\n")

p_glued = re.compile(r'([a-zA-Z])(' + kw + r')\s*(' + target + r')\b')

preamble_ok_rows = df_v2[df_v2['status'] == 'preamble_ok']
found_headings = []

for _, r in preamble_ok_rows.iterrows():
    c = r['Country']
    txt = str(r['extracted_text'])
    words = txt.split()
    if len(words) <= 30:
        continue
    prefix_30 = ' '.join(words[:30])
    rest = txt[len(prefix_30):]
    
    m_s = p_detector.search(rest)
    m_g = p_glued.search(rest)
    
    if m_s or m_g:
        pos = len(prefix_30) + (m_s.start(1) if m_s else m_g.start(2))
        heading_text = m_s.group(1) if m_s else (m_g.group(2) + " " + m_g.group(3))
        ext = txt[:pos].strip()
        w_before = len(words)
        w_after = len(ext.split())
        before_100 = ext[-100:] if len(ext) >= 100 else ext
        after_100 = txt[pos:pos+100]
        found_headings.append({
            'Country': c,
            'Heading': heading_text,
            'Position': pos,
            'Words Before': w_before,
            'Words After': w_after,
            'Before 100': before_100,
            'After 100': after_100
        })

print(f"Total heading matches found in preamble_ok: {len(found_headings)}")
if found_headings:
    for h in found_headings:
        print(f"Country: {h['Country']} | Heading: {h['Heading']} | Pos: {h['Position']}")
        print(f"  Words before cut: {h['Words Before']} -> after cut: {h['Words After']}")
        print(f"  100 chars BEFORE: {repr(h['Before 100'])}")
        print(f"  100 chars AFTER:  {repr(h['After 100'])}\n")
else:
    print("All 150 preamble_ok texts are clean of substantive headings after word 30.")


=== HEADING SCAN ACROSS ALL PREAMBLE_OK TEXTS ===

Total heading matches found in preamble_ok: 0
All 150 preamble_ok texts are clean of substantive headings after word 30.


## 6.3 Part C: Regression Fixes & Deep Ingestion Audits

We implement the specific structural and textual corrections:
1. **Norway Fix**: A capitalized heading keyword within the first 15 words (`Article The`) classifies Norway as a structural start. Because $7,231 > 1,000$ words, Norway returns to `no_preamble_in_dataset`. We confirm that Tajikistan, Tonga, and Liechtenstein stay non-structural.
2. **Australia Extraction**: Isolate the 70-word covering clause preamble (`WHEREAS the people of New South Wales... possessions of the Queen:`), dropping imperial Act titles. Status: `preamble_cut`.
3. **Canada Extraction**: Concatenate the 1867 British North America Act preamble (131 words) with the 1982 Charter of Rights preamble (17 words) to form a unified 148-word text. Status: `preamble_cut`.
4. **Canada Discrepancy & Full Word-Count Audit**: Print `doc_type_audit.csv` content for Canada and compare word counts across all 193 countries with `constitution_dataset_cleaned.csv`.

In [18]:
print("=== C.1 NORWAY STRUCTURAL START RE-EVALUATION ===")
cap_heading_words = {'CHAPTER', 'Chapter', 'PART', 'Part', 'TITLE', 'Title', 'ARTICLE', 'Article', 'Art', 'SECTION', 'Section', 'BOOK', 'Book'}

for c in ['Norway', 'Tajikistan', 'Tonga', 'Liechtenstein', 'Syria']:
    txt = str(df.loc[df['Country'] == c, 'cleaned_raw'].values[0])
    first_15_words = txt.split()[:15]
    has_cap_kw = any(re.sub(r'^[^\w]+|[^\w]+$', '', w) in cap_heading_words for w in first_15_words)
    print(f"Country: {c:<14} | First 15 words: {' '.join(first_15_words)}")
    print(f"  Has capitalized heading keyword: {has_cap_kw}")
print("Norway correctly returns to no_preamble_in_dataset; Tajikistan, Tonga, Liechtenstein stay non-structural.\n")

print("=== C.2 AUSTRALIA EXTRACTED PREAMBLE ===")
aus_txt = str(df.loc[df['Country'] == 'Australia', 'cleaned_raw'].values[0])
aus_start = aus_txt.find('WHEREAS the people of New South Wales')
aus_end = aus_txt.find('possessions of the Queen:') + len('possessions of the Queen:')
aus_extract = aus_txt[aus_start:aus_end].strip()
print(f"Australia extracted words: {len(aus_extract.split())}")
print(f"Text:\n{aus_extract}\n")

print("=== C.3 CANADA JOINED PREAMBLE ===")
can_txt = str(df.loc[df['Country'] == 'Canada', 'cleaned_raw'].values[0])
p1_start = 'WHEREAS the Provinces of Canada'
p1_end = 'British North America:'
can_p1_start = can_txt.find(p1_start)
can_p1_end = can_txt.find(p1_end, can_p1_start) + len(p1_end)
can_part_a = can_txt[can_p1_start:can_p1_end].strip()
can_part_b = 'Whereas Canada is founded upon principles that recognize the supremacy of God and the rule of law:'
can_joined = can_part_a + ' ' + can_part_b

print(f"1867 Preamble Part (a): {len(can_part_a.split())} words")
print(f"1982 Charter Part (b):  {len(can_part_b.split())} words")
print(f"Total joined preamble:  {len(can_joined.split())} words")
print(f"Joined Text:\n{can_joined}\n")

print("=== C.4 CANADA DISCREPANCY & DATASET WORD-COUNT AUDIT ===")
df_audit = pd.read_csv('../data/processed/doc_type_audit.csv')
can_audit_row = df_audit[df_audit['Country'] == 'Canada'].iloc[0]
print(f"doc_type_audit.csv for Canada:")
print(f"  words: {can_audit_row['words']}")
print(f"  doc_type: {can_audit_row['doc_type']}")
print(f"  first_150_chars: {repr(can_audit_row['first_150_chars'])}\n")

# Complete audit comparison against cleaned CSV
def normalize_spaces(text):
    if pd.isna(text):
        return ''
    return re.sub(r'\s+', ' ', str(text)).strip()

audit_mismatches = []
for _, r_aud in df_audit.iterrows():
    c = r_aud['Country']
    w_aud = r_aud['words']
    c_clean_row = df_clean[df_clean['Country'] == c]
    if len(c_clean_row) == 0:
        continue
    w_clean = len(normalize_spaces(c_clean_row.iloc[0]['Preamble']).split())
    if w_aud != w_clean:
        audit_mismatches.append({'Country': c, 'Words in Audit': w_aud, 'Words in Cleaned CSV': w_clean})

print(f"Word-count mismatches between doc_type_audit.csv and constitution_dataset_cleaned.csv: {len(audit_mismatches)}")
display(pd.DataFrame(audit_mismatches))
print("Finding: doc_type_audit.csv and constitution_dataset_cleaned.csv have identical word counts for 192/193 countries.")
print("The sole mismatch is Slovakia (19,651 in audit vs 0 in cleaned CSV), where raw text was patched in-memory from Notebook 01 Cell 31.")


=== C.1 NORWAY STRUCTURAL START RE-EVALUATION ===
Country: Norway         | First 15 words: A Form of government and religion Article The Kingdom of Norway is a free, independent,
  Has capitalized heading keyword: True
Country: Tajikistan     | First 15 words: WE, THE PEOPLE OF TAJIKISTAN, As an inseparable part of the world community, Realizing our
  Has capitalized heading keyword: False
Country: Tonga          | First 15 words: Granted by His Majesty King George Tupou I on the fourth day of November One
  Has capitalized heading keyword: False
Country: Liechtenstein  | First 15 words: We, John II, by the Grace of God, Prince Regnant of Liechtenstein, Duke of Troppau,
  Has capitalized heading keyword: False
Country: Syria          | First 15 words: Arab civilization, which is part of human heritage, has faced through its long history great
  Has capitalized heading keyword: False
Norway correctly returns to no_preamble_in_dataset; Tajikistan, Tonga, Liechtenstein stay non-structura

Word-count mismatches between doc_type_audit.csv and constitution_dataset_cleaned.csv: 1


,Country,Words in Audit,Words in Cleaned CSV
0,Slovakia,19651,0


Finding: doc_type_audit.csv and constitution_dataset_cleaned.csv have identical word counts for 192/193 countries.
The sole mismatch is Slovakia (19,651 in audit vs 0 in cleaned CSV), where raw text was patched in-memory from Notebook 01 Cell 31.


## 6.4 Part D: Final Extraction Decisions & V3 Dataset Generation

We finalize the dataset transformations:
1. **Lebanon**: Apply cut before `"Chapter One. The State and its Territory"` and strip the leading duplicated heading `"PART I BASIC PROVISIONS PART I BASIC PROVISIONS to the Constitution"`. Status: `preamble_cut`.
2. **Jordan**: Status `formula_only`.
3. **Costa Rica, Peru, Panama**: Retained as `preamble_ok`. Add `short_flag` boolean column (`extracted_words < 60`) for all rows.
4. **Strip Leading Labels**: Strip lone leading `"Preamble"`, `"PREAMBLE"`, or `"Foreword"` labels.
5. **Rename Status**: Change `no_preamble_in_source` to `no_preamble_in_dataset` across all records.
6. **Export V3**: Save to `data/processed/preamble_extraction_v3.csv`.

In [19]:
print("=== D.1 LEBANON CUT & CLEANING ===")
leb_raw = str(df.loc[df['Country'] == 'Lebanon', 'cleaned_raw'].values[0])
leb_cut_target = 'Chapter One. The State and its Territory'
idx_leb_cut = leb_raw.find(leb_cut_target)
if idx_leb_cut == -1:
    idx_leb_cut = leb_raw.find('Chapter One')
leb_cut_text = leb_raw[:idx_leb_cut].strip()

leb_dup_head = 'PART I BASIC PROVISIONS PART I BASIC PROVISIONS to the Constitution'
leb_final_text = leb_cut_text.replace(leb_dup_head, '').strip()

print(f"Lebanon Final Word Count: {len(leb_final_text.split())} words")
print(f"First 200 chars:\n  {repr(leb_final_text[:200])}")
print(f"Last 200 chars:\n  {repr(leb_final_text[-200:])}\n")

print("=== D.3 COSTA RICA, PERU, PANAMA FULL TEXT ===")
for sc in ['Costa Rica', 'Peru', 'Panama']:
    txt_sc = str(df.loc[df['Country'] == sc, 'cleaned_raw'].values[0])
    print(f"Country: {sc} ({len(txt_sc.split())} words)")
    print(f"  Full Text: {repr(txt_sc)}\n")

# Build V3 dataset
v3_status = []
v3_rule = []
v3_ext_text = []
v3_ext_words = []
v3_notes = []

stripped_labels_countries = []

for idx, r in df.iterrows():
    c = r['Country']
    txt = r['cleaned_raw']
    w = r['orig_words']
    v2_st = df_v2.loc[idx, 'status']
    v2_ext = str(df_v2.loc[idx, 'extracted_text'])
    v2_rl = df_v2.loc[idx, 'rule']
    v2_nt = df_v2.loc[idx, 'notes']
    
    if c == 'Australia':
        ext = aus_extract
        st = 'preamble_cut'
        rl = 'extract_covering_clause_preamble'
        notes = 'extracted covering clause preamble (dropped imperial Act title)'
    elif c == 'Canada':
        ext = can_joined
        st = 'preamble_cut'
        rl = 'join_1867_and_1982_preambles'
        notes = '1867 preamble + 1982 Charter preamble, joined'
    elif c == 'Lebanon':
        ext = leb_final_text
        st = 'preamble_cut'
        rl = 'cut_before_chapter_one_and_strip_dup_heading'
        notes = 'cut before Chapter One; stripped leading duplicated heading'
    elif c == 'Jordan':
        ext = txt
        st = 'formula_only'
        rl = 'promulgation_formula_only'
        notes = 'promulgation formula only (< 50 words); excluded from clustering'
    elif c == 'Norway':
        ext = ''
        st = 'no_preamble_in_dataset'
        rl = 'structural_start_and_words_gt_1000'
        notes = 'structural start (A Form of government... Article The); no preamble in dataset'
    elif v2_st == 'no_preamble_in_source':
        ext = ''
        st = 'no_preamble_in_dataset'
        rl = v2_rl
        notes = v2_nt.replace('no_preamble_in_source', 'no_preamble_in_dataset')
    else:
        ext = v2_ext
        st = v2_st
        rl = v2_rl
        notes = v2_nt
        
    # D.4 Strip lone leading label 'Preamble', 'PREAMBLE' or 'Foreword'
    if ext and ext != 'nan':
        m_lbl = re.match(r'^(Preamble|PREAMBLE|Foreword)\b[:\s-]*', ext)
        if m_lbl:
            ext = ext[m_lbl.end():].strip()
            notes += '; stripped lone leading label'
            stripped_labels_countries.append((c, m_lbl.group(0).strip()))
            
    ew = len(ext.split()) if ext and ext != 'nan' else 0
    if ew == 0:
        ext = ''
        
    v3_status.append(st)
    v3_rule.append(rl)
    v3_ext_text.append(ext)
    v3_ext_words.append(ew)
    v3_notes.append(notes)

df_v3 = df[['Country', 'orig_cluster', 'orig_words']].copy()
df_v3['status'] = v3_status
df_v3['rule'] = v3_rule
df_v3['extracted_words'] = v3_ext_words
df_v3['extracted_text'] = v3_ext_text
df_v3['notes'] = v3_notes
df_v3['short_flag'] = df_v3['extracted_words'] < 60

v3_csv_path = os.path.join(data_dir, 'preamble_extraction_v3.csv')
df_v3.to_csv(v3_csv_path, index=False)
print(f"Saved V3 extraction dataset to: {v3_csv_path}")
print(f"Total rows in V3: {len(df_v3)}")
print(f"Countries with stripped leading labels: {stripped_labels_countries}\n")


=== D.1 LEBANON CUT & CLEANING ===
Lebanon Final Word Count: 262 words
First 200 chars:
  'Lebanon is a sovereign, free, and independent homeland. An ultimate homeland for all its sons; unitary in terms of land, people, and institutions, within its boundaries; as provided in this Constituti'
Last 200 chars:
  'the law. No segregation of the people on any belonging whatsoever. No segregation, no partition, and no inhabitation. There is no legitimacy to any authority contradicting the charter of co-existence.'

=== D.3 COSTA RICA, PERU, PANAMA FULL TEXT ===
Country: Costa Rica (33 words)
  Full Text: 'We, the Representatives of the People of Costa Rica, freely elected Deputies to the National Constituent Assembly, invoking the name of God and reiterating our faith in Democracy, decree and sanction the following:'

Country: Peru (33 words)
  Full Text: 'The Democratic Constituent Congress invoking Almighty God, obeying the mandate of the Peruvian people, and remembering the sacrifice of a

## 6.5 Part E: Notebook 01 Provenance Audit

We inspect `notebooks/01_data_preprocessing.ipynb` to identify all cells assigning hand-pasted constitutions, recording cell index, country name, and raw pasted word count.

In [20]:
print("=== NOTEBOOK 01 PROVENANCE AUDIT (HAND-PASTED CONSTITUTIONS) ===\n")

with open('01_data_preprocessing.ipynb', 'r', encoding='utf-8') as f:
    nb01_audit = json.load(f)

pasted_cells = []

for i in range(4, 40):
    src = ''.join(nb01_audit['cells'][i].get('source', []))
    lines = [l.strip() for l in src.split('\n') if l.strip()]
    
    m_var = re.search(r'^([a-zA-Z0-9_]+)\s*=\s*\"\"\"', src, re.MULTILINE)
    var_name = m_var.group(1) if m_var else 'N/A'
    
    df_line = [l for l in lines if 'df.loc' in l][0]
    m1 = re.search(r"==\s*['\"]([^'\"]+)['\"]", df_line)
    m2 = re.search(r"['\"]Country['\"]\s*:\s*['\"]([^'\"]+)['\"]", df_line)
    country_name = m1.group(1) if m1 else (m2.group(1) if m2 else 'Unknown')
            
    m_str = re.search(r'\"\"\"(.*?)\"\"\"', src, re.DOTALL)
    if not m_str:
        m_str = re.search(r'\"\"\"\"(.*?)\"\"\"\"', src, re.DOTALL)
    pasted_text = m_str.group(1) if m_str else ''
    w_count = len(pasted_text.split())
    
    pasted_cells.append({
        'Cell Index': i,
        'Country': country_name,
        'Variable': var_name,
        'Pasted Words': w_count,
        'Assignment Method': 'df.loc[df["Country"] == ...]' if i <= 35 else 'df.loc[len(df)] (Appended)'
    })

display(pd.DataFrame(pasted_cells))


=== NOTEBOOK 01 PROVENANCE AUDIT (HAND-PASTED CONSTITUTIONS) ===



,Cell Index,Country,Variable,Pasted Words,Assignment Method
0,4,Australia,australia,16373,"df.loc[df[""Country""] == ...]"
1,5,Austria,austria,48476,"df.loc[df[""Country""] == ...]"
2,6,Botswana,botswana,30958,"df.loc[df[""Country""] == ...]"
3,7,Canada,canada,20341,"df.loc[df[""Country""] == ...]"
4,8,Cyprus,cyprus,35007,"df.loc[df[""Country""] == ...]"
5,9,Denmark,denmark,6063,"df.loc[df[""Country""] == ...]"
6,10,Finland,finland,13001,"df.loc[df[""Country""] == ...]"
7,11,France,france,13756,"df.loc[df[""Country""] == ...]"
8,12,Israel,israel,17774,"df.loc[df[""Country""] == ...]"
9,13,Italy,italy,11462,"df.loc[df[""Country""] == ...]"


## 6.6 Part F: Final Summary Statistics & Crosstab

We present the finalized summary statistics for `data/processed/preamble_extraction_v3.csv`:
- Status counts.
- Clusterable preambles (`preamble_ok` + `preamble_cut`).
- Word count statistics (min, median, mean, max, and count > 300 words).
- Crosstab of status by `orig_cluster`.

In [21]:
print("=== FINAL V3 STATUS VALUE COUNTS ===")
status_counts_v3 = df_v3['status'].value_counts()
display(pd.DataFrame({'Count': status_counts_v3, 'Percentage': (status_counts_v3 / len(df_v3) * 100).round(2)}))

clusterable_v3 = df_v3[df_v3['status'].isin(['preamble_ok', 'preamble_cut'])]
c_w3 = clusterable_v3['extracted_words']

print(f"\nTotal Clusterable Preambles (V3): {len(clusterable_v3)} countries")
print(f"Excluded from clustering: {len(df_v3) - len(clusterable_v3)} (29 no_preamble_in_dataset + 6 formula_only)")

print("\n=== CLUSTERABLE SET WORD COUNT STATISTICS (V3) ===")
stats_v3_df = pd.DataFrame({
    'Statistic': ['Count', 'Min Words', 'Median Words', 'Mean Words', 'Max Words', 'Exceeding 300 Words (>300)'],
    'Value': [len(clusterable_v3), c_w3.min(), c_w3.median(), round(c_w3.mean(), 1), c_w3.max(), int((c_w3 > 300).sum())]
})
display(stats_v3_df)

print("\n=== CROSSTAB OF FINAL V3 STATUS BY ORIGINAL CLUSTER ===")
crosstab_v3 = pd.crosstab(df_v3['status'], df_v3['orig_cluster'], margins=True)
display(crosstab_v3)


=== FINAL V3 STATUS VALUE COUNTS ===


,Count,Percentage
status,,
preamble_ok,149,77.20
no_preamble_in_dataset,29,15.03
preamble_cut,9,4.66
formula_only,6,3.11



Total Clusterable Preambles (V3): 158 countries
Excluded from clustering: 35 (29 no_preamble_in_dataset + 6 formula_only)

=== CLUSTERABLE SET WORD COUNT STATISTICS (V3) ===


,Statistic,Value
0,Count,158.0
1,Min Words,33.0
2,Median Words,203.0
3,Mean Words,298.4
4,Max Words,3223.0
5,Exceeding 300 Words (>300),50.0



=== CROSSTAB OF FINAL V3 STATUS BY ORIGINAL CLUSTER ===


orig_cluster,0,1,2,3,4,5,6,All
status,,,,,,,,
formula_only,1,1,1,0,1,2,0,6
no_preamble_in_dataset,0,0,7,0,10,12,0,29
preamble_cut,1,0,3,1,4,0,0,9
preamble_ok,24,16,14,19,26,14,36,149
All,26,17,25,20,41,28,36,193


# Section 7: Task 3d Validation and Final Dataset Lock

This section validates the extraction dataset V3 against disk, verifies provenance against source files, examines the historical audit discrepancies, audits the raw dataset for overwritten preambles, and locks the final dataset:
1. **Part A: Disk Validation**: Reloads `preamble_extraction_v3.csv` from disk, verifying row count, uniqueness, status counts, clusterable counts, empty text checks, and heading regex scans.
2. **Part B: Provenance of Every Extract**: Validates that all 158 clusterable texts originate as exact substrings of normalized source texts.
3. **Part C: Chile & Task 2c Audit**: Traces Chile across all iterations, reproduces Task 2c spot-checks from Notebook 02, and isolates the exact generation cells for Chile and Canada.
4. **Part D: Short Flag Recomputation**: Recomputes `short_flag` (`extracted_words < 60`) and catalogs all clusterable records below 60 words.
5. **Part E: Raw vs. Pasted Provenance Audit**: Loads original text from `data/raw/constitution_dataset.csv` for all 35 pasted countries plus New Zealand to verify whether real preambles were overwritten.
6. **Part F: Final Dataset Lock**: Exports `data/processed/preamble_final.csv` with locked columns: `Country, orig_cluster, status, extracted_text, extracted_words, short_flag, notes`.

## 7.1 Part A: Disk Validation & Integrity Assertions

We reload `data/processed/preamble_extraction_v3.csv` from disk and verify:
- Total rows: 193.
- Unique country count: 193.
- Status distribution: `preamble_ok` (149), `no_preamble_in_dataset` (29), `preamble_cut` (9), `formula_only` (6).
- Total clusterable preambles (`preamble_ok` + `preamble_cut`): 158.
- Empty extracted text count among clusterable preambles: 0.
- Residual heading matches among clusterable preambles: 0.

In [22]:
print("=== DISK VALIDATION: RELOADING PREAMBLE_EXTRACTION_V3.CSV ===\n")

v3_path = '../data/processed/preamble_extraction_v3.csv'
df_disk_v3 = pd.read_csv(v3_path)

row_count = len(df_disk_v3)
unique_countries = df_disk_v3['Country'].nunique()
status_counts_disk = df_disk_v3['status'].value_counts()
clusterable_mask = df_disk_v3['status'].isin(['preamble_ok', 'preamble_cut'])
df_clusterable_disk = df_disk_v3[clusterable_mask]
n_clusterable = len(df_clusterable_disk)

empty_clusterable = df_clusterable_disk[df_clusterable_disk['extracted_text'].isna() | (df_clusterable_disk['extracted_text'].str.strip() == '')]
n_empty_clusterable = len(empty_clusterable)

# Heading detector scan on clusterable texts
kw = r'(?:CHAPTER|Chapter|PART|Part|TITLE|Title|ARTICLE|Article|Art|SECTION|Section|BOOK|Book)'
target = r'(?:\d+|[IVXLCDM]+|[A-Z][a-zA-Z]*|first|second|third|fourth|fifth|One|Two|Three|Four|Five|Six|Seven|Eight|Nine|Ten)'
p_std = re.compile(r'(?:^|[.:;!?]\s+|(?:' + kw + r'\s+' + target + r'[\s.:]+))(' + kw + r'\s+' + target + r')\b')
p_glu = re.compile(r'([a-zA-Z])(' + kw + r')\s*(' + target + r')\b')

heading_matches_clusterable = []
for _, r in df_clusterable_disk.iterrows():
    c = r['Country']
    txt = str(r['extracted_text'])
    words = txt.split()
    if len(words) <= 30:
        continue
    prefix_30 = ' '.join(words[:30])
    rest = txt[len(prefix_30):]
    m_s = p_std.search(rest)
    m_g = p_glu.search(rest)
    if m_s or m_g:
        heading_matches_clusterable.append((c, m_s.group(1) if m_s else (m_g.group(2) + " " + m_g.group(3))))

n_heading_matches = len(heading_matches_clusterable)

print(f"Row count:                       {row_count}")
print(f"Unique countries:                {unique_countries}")
print(f"Status distribution:\n{status_counts_disk}")
print(f"Total clusterable preambles:     {n_clusterable}")
print(f"Clusterable rows with empty text:{n_empty_clusterable}")
print(f"Clusterable texts with headings: {n_heading_matches}")

# Assertions
assert row_count == 193, f"Assertion failed: expected 193 rows, got {row_count}"
assert unique_countries == 193, f"Assertion failed: expected 193 unique countries, got {unique_countries}"
assert status_counts_disk['preamble_ok'] == 149, f"Assertion failed: preamble_ok count {status_counts_disk['preamble_ok']}"
assert status_counts_disk['no_preamble_in_dataset'] == 29, f"Assertion failed: no_preamble count {status_counts_disk['no_preamble_in_dataset']}"
assert status_counts_disk['preamble_cut'] == 9, f"Assertion failed: preamble_cut count {status_counts_disk['preamble_cut']}"
assert status_counts_disk['formula_only'] == 6, f"Assertion failed: formula_only count {status_counts_disk['formula_only']}"
assert n_clusterable == 158, f"Assertion failed: expected 158 clusterable, got {n_clusterable}"
assert n_empty_clusterable == 0, f"Assertion failed: expected 0 empty clusterable, got {n_empty_clusterable}"
assert n_heading_matches == 0, f"Assertion failed: expected 0 heading matches, got {n_heading_matches} ({heading_matches_clusterable})"

print("\nAll Disk Validation Assertions PASSED successfully (193, 193, 149/29/9/6, 158, 0, 0).")


=== DISK VALIDATION: RELOADING PREAMBLE_EXTRACTION_V3.CSV ===

Row count:                       193
Unique countries:                193
Status distribution:
status
preamble_ok               149
no_preamble_in_dataset     29
preamble_cut                9
formula_only                6
Name: count, dtype: int64
Total clusterable preambles:     158
Clusterable rows with empty text:0
Clusterable texts with headings: 0

All Disk Validation Assertions PASSED successfully (193, 193, 149/29/9/6, 158, 0, 0).


## 7.2 Part B: Provenance of Every Extract

We verify that every clusterable text in `preamble_extraction_v3.csv` is an exact substring of the normalized source text:
- Source: `data/processed/constitution_dataset_cleaned.csv` (column `'Preamble'`), normalized for HTML entities and whitespace.
- Slovakia: verified against `slovak_republic` string from Notebook 01 Cell 31.
- Established exceptions:
  - Canada: two joined preambles (1867 Act + 1982 Charter).
  - France, Libya: stripped leading `'Preamble '` label.
  - Croatia: stripped duplicated leading heading.
  - Lebanon: stripped duplicated leading heading and cut before `Chapter One`.
  - Cuba: dropped trailing `': CONSTITUTION'` label.

In [23]:
print("=== PROVENANCE OF EVERY EXTRACT (158 CLUSTERABLE TEXTS) ===\n")

df_clean_source = pd.read_csv('../data/processed/constitution_dataset_cleaned.csv')
df_clean_source = df_clean_source.drop(index=192).reset_index(drop=True)

with open('01_data_preprocessing.ipynb', 'r', encoding='utf-8') as f:
    nb01_prov = json.load(f)
loc_prov = {}
exec(''.join(nb01_prov['cells'][31]['source']), {'df': pd.DataFrame([{'Country': 'Slovakia', 'Preamble': None}])}, loc_prov)
df_clean_source.loc[df_clean_source['Country'] == 'Slovakia', 'Preamble'] = loc_prov['slovak_republic']

def normalize_text(text):
    if pd.isna(text):
        return ''
    t = html.unescape(str(text))
    t = t.replace('x27;', "'").replace('quot;', '"').replace('nbsp;', ' ').replace('amp;', '&')
    t = re.sub(r'\bgt;', ' ', t)
    t = re.sub(r'\blt;', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

df_clean_source['normalized_raw'] = df_clean_source['Preamble'].apply(normalize_text)

provenance_mismatches = []
known_transformations = {'Canada', 'France', 'Libya', 'Croatia', 'Lebanon', 'Cuba'}

for _, r in df_clusterable_disk.iterrows():
    c = r['Country']
    ext = str(r['extracted_text'])
    source_row = df_clean_source[df_clean_source['Country'] == c]
    if len(source_row) == 0:
        provenance_mismatches.append((c, 'Not found in source', ''))
        continue
    source_txt = str(source_row.iloc[0]['normalized_raw'])
    
    is_exact_sub = ext in source_txt
    if not is_exact_sub:
        if c == 'Canada':
            p1_in = 'WHEREAS the Provinces of Canada' in source_txt and 'British North America:' in source_txt
            p2_in = 'Whereas Canada is founded upon principles that recognize the supremacy of God and the rule of law:' in source_txt
            if not (p1_in and p2_in):
                provenance_mismatches.append((c, 'Canada parts missing from source', ''))
        elif c in ['France', 'Libya']:
            if ('Preamble ' + ext not in source_txt) and (ext not in source_txt):
                provenance_mismatches.append((c, f"{c} leading label transformation mismatch", ''))
        elif c == 'Croatia':
            if ('I Historical Foundations I Historical Foundations ' + ext not in source_txt) and (ext not in source_txt):
                provenance_mismatches.append((c, 'Croatia heading strip mismatch', ''))
        elif c == 'Lebanon':
            dup_h = 'PART I BASIC PROVISIONS PART I BASIC PROVISIONS to the Constitution '
            if (dup_h + ext not in source_txt) and (ext not in source_txt):
                provenance_mismatches.append((c, 'Lebanon heading strip mismatch', ''))
        elif c == 'Cuba':
            if (ext + ': CONSTITUTION' not in source_txt) and (ext + ' CONSTITUTION' not in source_txt):
                provenance_mismatches.append((c, 'Cuba trailing label strip mismatch', ''))
        else:
            provenance_mismatches.append((c, ext[:120], source_txt[:120]))

print(f"Total provenance mismatches (excluding verified structural transformations): {len(provenance_mismatches)}")
assert len(provenance_mismatches) == 0, f"Provenance validation failed for: {provenance_mismatches}"
print("All 158 clusterable extracts verified as 100% authentic substrings of source text.")


=== PROVENANCE OF EVERY EXTRACT (158 CLUSTERABLE TEXTS) ===



Total provenance mismatches (excluding verified structural transformations): 0
All 158 clusterable extracts verified as 100% authentic substrings of source text.


## 7.3 Part C: Chile Evolution & Task 2c Audit Reproduction

We trace the textual evolution of Chile across all 6 files, reproduce the exact spot-check cell from Notebook 02, and document the cell code generating the values in Task 2c for Chile and Canada.

In [24]:
print("=== CHILE ACROSS ALL ITERATIONS ===\n")

chile_iterations = [
    ('Raw Dataset CSV', '../data/raw/constitution_dataset.csv', 'Preamble'),
    ('Cleaned Dataset CSV', '../data/processed/constitution_dataset_cleaned.csv', 'Preamble'),
    ('Clustered Dataset CSV', '../data/processed/constitution_preamble_clustered.csv', 'clean_preamble_text'),
    ('Proposal CSV', '../data/processed/preamble_extraction_proposal.csv', 'extracted_text'),
    ('Dataset V2 CSV', '../data/processed/preamble_extraction_v2.csv', 'extracted_text'),
    ('Dataset V3 CSV', '../data/processed/preamble_extraction_v3.csv', 'extracted_text'),
]

for label, p, col in chile_iterations:
    df_it = pd.read_csv(p)
    chile_val = str(df_it[df_it['Country'] == 'Chile'][col].values[0])
    w = len(chile_val.split())
    print(f"[{label}] (Column: {col})")
    print(f"  Word count:     {w}")
    print(f"  First 200 chars:{repr(chile_val[:200])}\n")

print("=== REPRODUCING NOTEBOOK 02 TASK 2c SPOT-CHECK CELL ===\n")
with open('02_embedding_comparison.ipynb', 'r', encoding='utf-8') as f:
    nb02_spot = json.load(f)

# Cell 24 saved output text
cell_24_output_text = ''.join(nb02_spot['cells'][24]['outputs'][0]['text'])
cell_24_plain_text = ''.join(nb02_spot['cells'][24]['outputs'][1]['data']['text/plain'])
cell_24_sample_text = ''.join(nb02_spot['cells'][24]['outputs'][2]['text'])
cell_24_app_plain = ''.join(nb02_spot['cells'][24]['outputs'][3]['data']['text/plain'])

print("Notebook 02 Cell 24 Saved Output (Sample from app.py):\n")
for line in cell_24_app_plain.split('\n')[:15]:
    print("  ", line)

# Compare spot-check rows against doc_type_audit.csv and cleaned CSV
df_audit_compare = pd.read_csv('../data/processed/doc_type_audit.csv')
spot_check_countries = [
    'Australia', 'Canada', 'United Kingdom', 'France', 'Mexico',
    'South Africa', 'Kenya', 'Congo', 'Ethiopia',
    'Argentina', 'Bolivia', 'Brazil', 'Chile', 'United States of America',
    'India', 'Ukraine', 'Russia', 'Estonia',
    'Afghanistan', 'Pakistan', 'Iran', 'Iraq', 'Egypt',
    'Indonesia', 'Bangladesh', 'Bhutan', 'Thailand', 'China',
    'Germany', 'Austria', 'Belgium', 'Norway', 'Italy',
    'Czechia', 'Qatar', 'São Tomé and Principe', 'Mongolia', 'Slovenia',
    'Belarus', 'Benin', 'Monaco', 'Grenada', 'Greece', 'Somalia'
]

spot_mismatches = []
for c in set(spot_check_countries):
    r_aud = df_audit_compare[df_audit_compare['Country'] == c].iloc[0]
    r_cln = df_clean_source[df_clean_source['Country'] == c].iloc[0]
    
    w_aud = r_aud['words']
    w_cln = len(normalize_text(r_cln['Preamble']).split())
    first150_aud = str(r_aud['first_150_chars'])[:150]
    first150_cln = str(r_cln['Preamble'])[:150]
    
    if w_aud != w_cln:
        spot_mismatches.append({'Country': c, 'Field': 'Word Count', 'Audit': w_aud, 'Clean': w_cln})
    if first150_aud != first150_cln:
        spot_mismatches.append({'Country': c, 'Field': 'First 150 Chars', 'Audit': first150_aud, 'Clean': first150_cln})

print(f"\nTotal spot-check mismatches between doc_type_audit.csv and cleaned CSV: {len(spot_mismatches)}")
if spot_mismatches:
    display(pd.DataFrame(spot_mismatches))
else:
    print("Zero mismatches across all spot-checked countries.")


=== CHILE ACROSS ALL ITERATIONS ===

[Raw Dataset CSV] (Column: Preamble)
  Word count:     83
  First 200 chars:'We the people of Chile, responsible for her history and future, have drafted this Political Constitution of the Republic of Chile. We have done so in a free and democratic way, exercising the original'

[Cleaned Dataset CSV] (Column: Preamble)
  Word count:     83
  First 200 chars:'We the people of Chile, responsible for her history and future, have drafted this Political Constitution of the Republic of Chile. We have done so in a free and democratic way, exercising the original'

[Clustered Dataset CSV] (Column: clean_preamble_text)
  Word count:     35
  First 200 chars:'chile responsible history future drafted constitution republic chile free democratic way exercising original constituent power possess goal solemn live peace justice prosperity borders coexist countri'

[Proposal CSV] (Column: extracted_text)
  Word count:     83
  First 200 chars:'We the people of Chile

## 7.4 Part D: Short Flag Recomputation

We recompute `short_flag` (`extracted_words < 60`) directly from `preamble_extraction_v3.csv` and report all clusterable preambles with fewer than 60 words.

In [25]:
print("=== SHORT FLAG RECOMPUTATION (EXTRACTED_WORDS < 60) ===\n")

df_disk_v3['short_flag_recomp'] = df_disk_v3['extracted_words'] < 60
assert (df_disk_v3['short_flag_recomp'] == df_disk_v3['short_flag']).all(), "short_flag mismatch in file!"

short_clusterable = df_disk_v3[clusterable_mask & df_disk_v3['short_flag']].sort_values('extracted_words')

print(f"Total clusterable countries with short_flag == True (< 60 words): {len(short_clusterable)}\n")
display(short_clusterable[['Country', 'orig_cluster', 'status', 'extracted_words', 'notes']])


=== SHORT FLAG RECOMPUTATION (EXTRACTED_WORDS < 60) ===

Total clusterable countries with short_flag == True (< 60 words): 7



,Country,orig_cluster,status,extracted_words,notes
40,Costa Rica,0,preamble_ok,33,standard clean preamble (<= 400 words)
133,Peru,0,preamble_ok,33,standard clean preamble (<= 400 words)
130,Panama,0,preamble_ok,43,standard clean preamble (<= 400 words)
182,United States of America,0,preamble_ok,52,standard clean preamble (<= 400 words)
132,Paraguay,0,preamble_ok,53,standard clean preamble (<= 400 words)
55,Estonia,1,preamble_ok,57,standard clean preamble (<= 400 words)
85,Kazakhstan,1,preamble_ok,59,standard clean preamble (<= 400 words)


## 7.5 Part E: Raw vs. Pasted Provenance Audit

We inspect `data/raw/constitution_dataset.csv` for the 35 hand-pasted countries in Notebook 01 plus New Zealand to determine whether any original preamble was overwritten by the pasting operation.

In [26]:
print("=== RAW DATASET AUDIT ON 36 PASTED COUNTRIES ===\n")

df_raw_all = pd.read_csv('../data/raw/constitution_dataset.csv')

pasted_36 = [
    'Australia', 'Austria', 'Botswana', 'Canada', 'Cyprus', 'Denmark',
    'Finland', 'France', 'Israel', 'Italy', 'Jamaica', 'Lesotho',
    'Luxembourg', 'Malaysia', 'Maldives', 'Malta', 'Mauritius', 'Mexico',
    'Monaco', 'Netherlands', 'Norway', 'Oman', 'Qatar', 'Romania',
    'Saudi Arabia', 'Sierra Leone', 'Singapore', 'Slovakia', 'Somalia',
    'Suriname', 'Sweden', 'Uruguay', 'Libya', 'Palestine', 'Yemen',
    'New Zealand'
]

raw_audit_results = []
flagged_raw = []

for c in sorted(pasted_36):
    r_raw = df_raw_all[df_raw_all['Country'] == c]
    if len(r_raw) == 0:
        raw_audit_results.append({
            'Country': c,
            'Raw Words': 0,
            'First 120 Characters': 'Not present in raw dataset (appended in NB01)',
            'Flag': 'None'
        })
        continue
    raw_val = r_raw.iloc[0]['Preamble']
    if pd.isna(raw_val):
        w = 0
        snippet = 'nan (missing in raw dataset)'
        flag = 'None'
    else:
        w = len(str(raw_val).split())
        snippet = str(raw_val)[:120].replace('\n', ' ')
        # Flag condition: words <= 1000 and not structural start
        first_15 = str(raw_val).split()[:15]
        has_struct = any(re.sub(r'^[^\w]+|[^\w]+$', '', word) in cap_heading_words for word in first_15) or any(re.match(r'^[IVXLCDMivxlcdm]+[\.:]?$', word) for word in first_15)
        if w > 0 and w <= 1000 and not has_struct:
            flag = 'POTENTIAL PREAMBLE'
            flagged_raw.append(c)
        else:
            flag = 'None'
            
    raw_audit_results.append({
        'Country': c,
        'Raw Words': w,
        'First 120 Characters': snippet,
        'Flag': flag
    })

display(pd.DataFrame(raw_audit_results))
print(f"\nTotal countries flagged (<= 1000 words & non-structural in raw dataset): {len(flagged_raw)}")
print("Finding: 32 countries were NaN in raw CSV, 3 were not present in raw CSV, Canada was 5,434 words of statutory body text, and New Zealand was 3,419 words of table of contents.")
print("Zero genuine preambles were overwritten or masked by the Notebook 01 pasting operations.")


=== RAW DATASET AUDIT ON 36 PASTED COUNTRIES ===



,Country,Raw Words,First 120 Characters,Flag
0,Australia,0,nan (missing in raw dataset),None
1,Austria,0,nan (missing in raw dataset),None
2,Botswana,0,nan (missing in raw dataset),None
3,Canada,5434,CONSTITUTION ACT 1867 JUDICATURE; THE FIRST SCHEDULE. ELECTORAL DISTRICTS OF ONTARIO CONSTITUTION ACT 1867An Act for the,None
4,Cyprus,0,nan (missing in raw dataset),None
5,Denmark,0,nan (missing in raw dataset),None
6,Finland,0,nan (missing in raw dataset),None
7,France,0,nan (missing in raw dataset),None
8,Israel,0,nan (missing in raw dataset),None
9,Italy,0,nan (missing in raw dataset),None



Total countries flagged (<= 1000 words & non-structural in raw dataset): 0
Finding: 32 countries were NaN in raw CSV, 3 were not present in raw CSV, Canada was 5,434 words of statutory body text, and New Zealand was 3,419 words of table of contents.
Zero genuine preambles were overwritten or masked by the Notebook 01 pasting operations.


## 7.6 Part F: Final Dataset Lock

Having validated disk consistency (Part A) and text provenance (Part B), we save the locked final dataset to `data/processed/preamble_final.csv` with columns:
`Country, orig_cluster, status, extracted_text, extracted_words, short_flag, notes`.

In [27]:
print("=== LOCKING FINAL DATASET: DATA/PROCESSED/PREAMBLE_FINAL.CSV ===\n")

final_cols = ['Country', 'orig_cluster', 'status', 'extracted_text', 'extracted_words', 'short_flag', 'notes']
df_final = df_disk_v3[final_cols].copy()

final_csv_path = '../data/processed/preamble_final.csv'
df_final.to_csv(final_csv_path, index=False)
print(f"Successfully locked and saved final dataset to: {final_csv_path}")
print(f"Total rows: {len(df_final)}")
print(f"Columns:    {df_final.columns.tolist()}\n")

# Verify reloaded final CSV
df_check_final = pd.read_csv(final_csv_path)
assert len(df_check_final) == 193, "Final CSV row count assertion failed!"
assert (df_check_final.columns == final_cols).all(), "Final CSV columns assertion failed!"
print("Final dataset lock verified on disk.")


=== LOCKING FINAL DATASET: DATA/PROCESSED/PREAMBLE_FINAL.CSV ===

Successfully locked and saved final dataset to: ../data/processed/preamble_final.csv
Total rows: 193
Columns:    ['Country', 'orig_cluster', 'status', 'extracted_text', 'extracted_words', 'short_flag', 'notes']

Final dataset lock verified on disk.
